In [ ]:

import numpy as np
from typing import Callable
from scipy.spatial.distance import cdist
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

class EvolvedCANN(object):
    
    def __init__(self):
        # Network basic parameters
        self._nn = 1000 # Number of neurons.
        self._np = 250 # number of place fields.

        self._L = 8.88 # Length of the track (meters).
        self._rho = self._np / self._L # Density of place fields (fields per meter).
        self._field_size_sigma = 0.2 # Standard deviation of place fields.
        
        # Initialize place fields for map 1 and map 2.
        self._pc_index = np.sort(np.random.choice(
            self._nn, 
            size=self._np, 
            replace=False
        )) # Place cell indices.
        self._x_centers_m1 = np.zeros(self._nn, np.float64) - np.inf # Place field centers.
        self._x_centers_m1[self._pc_index] = np.linspace(0, self._L, self._np+1)[:-1] + self._L/(2*self._np)
        
        # Retrieval Sharpness
        self._rp_sharpness = 20
        # Correlation threshold
        self._corr_thre = 0.3
        
        # Noise distribution
        self._weight_noise_sigma = 3e-6
        self._input_noise_sigma = 3e-6

        # Weight matrix and BTSP features
        self._temporal_sigma = 2 # seconds
        self._vel_map1 = 0.2 # m/s
        self._activity_thre = 0.6 # Activity threshold for plateau initiation
        self._J0 = 5*1e-6 # Global inhibition
        self._J1 = 2e-5 # Excitation within place fields
        self._alpha = 10 # amplification factor for weight updates
        self._W = np.zeros((self._nn, self._nn)) # Weight matrix
        self._W0 = np.zeros((self._nn, self._nn)) - self._J0 # Weight matrix for map 1
        self._W1 = np.zeros((self._nn, self._nn)) - self._J0 # Initial weight matrix for map 2
        self._WI1 = np.zeros((self._nn, self._nn)) # Weight matrix for path integration input
        self._WI0 = np.zeros((self._nn, self._nn)) # Initial weight matrix for path integration input

        # Dynamical variables
        self._t = 0
        self._x = 0.0
        self._dt = 0.05 # seconds
        self._tau = 1.0
        self._c = 0.0 # Current correlation coefficient between PVr and PVm
        self._rp = 0.0 # Retrieval progress, [0, 1]
        self._u = np.zeros(self._nn) # Real-time population vector activity (PVr)
        self._v = np.zeros(self._nn) # PV anticipated retrieved (PVm)
        self._Iext = np.zeros(self._nn) # external input
        self._r = np.zeros(self._nn) # Tuning curve of all neurons, including non-place cells.
        self._AT = np.full(self._nn, -np.inf) # Activate Time of each neurons
    
        self._init_W()
    
    @property
    def nn(self):
        return self._nn
    @property
    def npc(self):
        return self._np
    @property
    def pc_index(self):
        return self._pc_index

    @property
    def m2_index(self):
        return self._m2_index

    @property
    def W(self):
        return self._W
        
    @property
    def m2_index(self):
        return self._m2_index
    
    def _init_u(self, x0):
        """Initial activity bump shares some similarity with the tuning curve
        of map 1 at x = 0."""
        x_fields_m2 = self._x_centers_m2
        self._u0 = np.exp(
            -0.5 * (x_fields_m2 - x0) ** 2 / self._field_size_sigma ** 2
        )

    def _init_W(self):
        """Initialize the weight matrix for map 1.
        """
        cdist_mat = cdist(
            self._x_centers_m1[self._pc_index, None], 
            self._x_centers_m1[self._pc_index, None], 
            metric='euclidean'
        )     
        #Ricker wavelet
        self._W0[np.ix_(self._pc_index, self._pc_index)] += np.exp(
            -0.5 * (cdist_mat / (self._temporal_sigma*self._vel_map1)) ** 2
        ) * self._J1
        self._W = (
            self._W0.copy() + 
            self._W1.copy() + 
            self._J0 + 
            np.random.randn(self._nn, self._nn) * self._weight_noise_sigma
        )

    def _init_initial_states(self):
        self._m2_index = np.random.choice(
            self._nn, 
            size=self._np, 
            replace=False
        ) # Cell indices for new maps.
        self._x_centers_m2 = np.zeros(self.nn, np.float64) - np.inf
        self._x_centers_m2[self._m2_index] = np.linspace(0, self._L, self._np+1)[:-1] + self._L/(2*self._np)
        self.m2_activated = np.zeros(self._nn, dtype=bool)
        
        # Get Identify matrix
        pc = np.zeros((self._np, self._np))
        np.fill_diagonal(pc, 1)
        self._WI0[np.ix_(self._pc_index, self._pc_index)] = pc
        self._WI1[np.ix_(self._m2_index, self._pc_index)] = pc
        
    def T(self, x: float) -> np.ndarray:
        """Tuning curve of all neurons, including non-place cells.
        
        Parameters
        ----------
        x : float
            Animal's position on the track.
            
        Returns
        -------
        np.ndarray
            Tuning curve of all neurons, including non-place cells. Shape: (self._nn,). 
        """
        tuning_curve = np.exp(
            -0.5 * (self._x_centers_m1 - x) ** 2 / self._field_size_sigma ** 2
        )
        tuning_curve[np.isnan(tuning_curve)] = 0
        return tuning_curve
    
    def dT(self, x, dx=1e-3):
        """Numerical derivative of the tuning curve.
        """
        return (self.T(x + dx) - self.T(x - dx)) / (2 * dx)
        
    def sigmoid(self, x: np.ndarray) -> np.ndarray:
        """Sigmoid activation function
        """
        return 1 / (1 + np.exp(-x+0.2))

    def update_corr(self, u: np.ndarray, v: np.ndarray) -> float:
        """Check if the network is retrieving the map 1.

        Parameters
        ----------
        u : np.ndarray
            Real-time population vector activity (PVr).
        v : np.ndarray
            PV anticipated retrieved (PVm).

        Returns
        -------
        bool
            True if the network is retrieving the map 1, False otherwise.
        """
        pc_index = self._pc_index
        return np.corrcoef(u[pc_index], v[pc_index])[0, 1]

    def update_rp(self, corr: float) -> float:
        """Logistic function to update the probability of dendritic plateaus,
        depending on the degree of retrievals.
        
        Parameters
        ----------
        corr : float
            Pearson correlation coefficient between PVr and PVm.

        Returns
        -------
        float
            Updated probability of dendritic plateaus.
        """
        _rp = 1 / (1 + np.exp(-self._rp_sharpness * (corr - self._corr_thre)))
        return _rp

    def update_W(self):
        """Update the recurrent weight matrix with BTSP-like learning rule.
        """
        AT = self._AT
        idx = np.where(AT > -1e-2)[0]
        
        if len(idx) < 2:
            return self._W
        # Ricker wavelet
        time_intervals = cdist(AT[idx, None], AT[idx, None], metric='euclidean')

        self._W1[np.ix_(idx, idx)] = np.exp(
            -0.5 * (time_intervals / self._temporal_sigma) ** 2
        ) * self._J1 - self._J0
        self._W = (
            self._W0 + 
            self._W1 + 
            self._J0 + 
            np.random.randn(self._nn, self._nn) * self._weight_noise_sigma
        )
        return self._W

    def update_WI(self):
        """Update the weight matrix for path integration input.
        """
        WI = (
            (1-self._rp) * self._WI1 + 
            self._rp * self._WI0
        )
        return WI

    def simulate(
        self, 
        x0: float, 
        vel: float=0.10, 
        n_steps: int=200, 
        n_trials: int=5
    ):
        """Simulate the network dynamics.

        Parameters
        ----------
        start_x : float
            Starting position of the animal.
        vel : float, optional
            Velocity of the animal, by default 0.05 m/s.
        n_steps : int, optional
            Number of time steps to simulate, by default 200.
        """
        self._init_initial_states()
        self._init_u(x0)

        # Recording variables
        u_record = np.zeros((n_trials, int(n_steps/100), self._nn)) * np.nan
        v_record = np.zeros((n_trials, int(n_steps/100), self._nn)) * np.nan
        x_record = np.zeros((n_trials, int(n_steps/100))) * np.nan
        t_record = np.zeros((n_trials, int(n_steps/100))) * np.nan
        corr_record = np.zeros((n_trials, int(n_steps/100))) * np.nan
        rp_record = np.zeros((n_trials, int(n_steps/100))) * np.nan
        Iext_record = np.zeros((n_trials, int(n_steps/100), self._nn)) * np.nan
        W_record = np.zeros((n_trials, int(n_steps/100), self._nn, self._nn)) * np.nan

        for trial in range(n_trials):
            self._x = x0
            self._t = 0
            self._u = self._u0
            self._v = self.T(x0)
            self._c = self.update_corr(self._u, self._v)
            self._rp = self.update_rp(self._c)
            self._AT = np.full(self._nn, -np.inf)
            WI = self.update_WI()
            
            u_record[trial, 0] = self._u
            v_record[trial, 0] = self._v
            x_record[trial, 0] = self._x
            t_record[trial, 0] = self._t
            corr_record[trial, 0] = self._c
            rp_record[trial, 0] = self._rp
            Iext_record[trial, 0] = self._Iext
            W_record[trial, 0] = self._W
            prev_cell = -1
            
            for s in range(1, n_steps):
                # Update position
                self._x += vel * self._dt
                self._t += self._dt
                if self._x > self._L:
                    break

                # Update path integration signals
                # Position changes update external input
                self._Iext = WI @ self.dT(self._x) + np.random.randn(self._nn) * self._input_noise_sigma
                
                # update_u
                W = self._W
                du = (-self._u + W @ self._u + self._Iext * vel)
                self._u += du * self._dt / self._tau
                self._u = np.clip(self._u, 0, 1)
                self._u = self._u / np.nanmax(self._u)
                self._u[np.isnan(self._u) | (self._u < 0)] = 0
                
                # update v
                self._v = self.T(self._x)
                
                # determine if the network is retrieving map 1
                self._c = self.update_corr(self._u, self._v)
                self._rp = self.update_rp(self._c)
                WI = self.update_WI()
                
                # update recurrent weights
                new_recruited = np.where(
                    (self._x_centers_m2 <= self._x) &
                    (self._x_centers_m2 >= x0)
                )[0]
                self.m2_activated[new_recruited] = 1
            
                if len(new_recruited) > 0:
                    new_recruited = new_recruited[np.argsort(self._x_centers_m2[new_recruited])]
                    is_plateau = np.random.choice([0, 1], p=[self._rp, 1-self._rp])
                    if (new_recruited[-1] != prev_cell) and (is_plateau == 1):
                        # Trigger BTSP-based learning of new place cells
                        self._AT[new_recruited[-1]] = self._t
                        self.update_W()
                        prev_cell = new_recruited[-1]
                        
                # Record activity
                if s % 100 != 0:
                    continue
                step = int(s / 100)
                u_record[trial, step] = self._u
                v_record[trial, step] = self._v
                x_record[trial, step] = self._x
                t_record[trial, step] = self._t
                corr_record[trial, step] = self._c
                rp_record[trial, step] = self._rp
                Iext_record[trial, step] = self._Iext
                W_record[trial, step] = self._W

        return (
            u_record, v_record, x_record, t_record, 
            corr_record, rp_record, Iext_record, W_record
        )
        
#CANN_model = EvolvedCANN()
#res = CANN_model.simulate(x0=5, vel=0.05, n_steps=3000, n_trials=1)